In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
from langchain_neo4j import Neo4jGraph

graph = Neo4jGraph(
    url=os.getenv("NEO4J_URI"),
    username=os.getenv("NEO4J_USERNAME"),  
    password=os.getenv("NEO4J_PASSWORD"),
) 

In [6]:
cypher_query = """
create (a:Person {name: 'Alice', age: 30})
return a
"""
graph.query(cypher_query)

[{'a': {'name': 'Alice', 'age': 30}}]

In [10]:
def reset_database(graph):
    """
    reset Database 
    """
    graph.query("MATCH (n) DETACH DELETE n")


  #delete all constraints
    constraints = graph.query("SHOW CONSTRAINTS")
    for constraint in constraints:
        constraint_name = constraint["name"]
        if constraint_name:
            graph.query(f"DROP CONSTRAINT {constraint_name}")

    # delete all indexes
    indexes = graph.query("SHOW INDEXES")
    for index in indexes:
        index_name = index.get("name")
        index_type = index.get("type")
        if index_name and index_type != "CONSTRAINT":
            graph.query(f"DROP INDEX {index_name}")
    
    print("Database reset complete.")

reset_database(graph)

Database reset complete.


In [11]:
cypher_query = """
CREATE (a:Person {name: 'Alice', age: 30})
"""
graph.query(cypher_query)

[]

In [12]:
cypher_query = """
CREATE (a:Person {name: 'Heide', age: 30})
CREATE (b:Person {name: 'Bob', age: 25})
CREATE (c:City {name: 'Seoul', population: 9733509})
"""
graph.query(cypher_query)

[]

In [13]:
cypher_query = """
MATCH (a:Person {name: 'Alice'}), (b:Person {name: 'Bob'})
CREATE (a)-[:KNOWS{since: 2020}]->(b)
"""
graph.query(cypher_query)

[]

In [15]:
cypher_query = """
CREATE (a:Person {name: 'Park'})-[:LIVES_IN]->(b:City {name: 'Manchester'})
return a, b
"""
graph.query(cypher_query)

[{'a': {'name': 'Park'}, 'b': {'name': 'Manchester'}}]

In [22]:
cypher_query = """
MATCH (a:Person {name: 'Alice'}),
      (c:City {name: 'Seoul'})
CREATE (a)-[r:LIVES_IN]->(c)
return a,r,c
"""
graph.query(cypher_query)

[{'a': {'name': 'Alice', 'age': 30},
  'r': ({'name': 'Alice', 'age': 30},
   'LIVES_IN',
   {'name': 'Seoul', 'population': 9733509}),
  'c': {'name': 'Seoul', 'population': 9733509}}]

In [25]:
cypher_query = """
MATCH (a:Person {name: 'Alice'}),
(b:Person {name: 'Heide'}),
(c:City {name: 'Seoul'})
CREATE (a)-[r:LIVES_IN]->(c),
        (a)-[r2:KNOWS]->(b),
        (a)<-[r3:KNOWS]-(b)
return a, b, c
"""
graph.query(cypher_query)

[{'a': {'name': 'Alice', 'age': 30},
  'b': {'name': 'Heide', 'age': 30},
  'c': {'name': 'Seoul', 'population': 9733509}}]